In [3]:
# test_SUBTRACT_MU.py

import numpy as np
from scipy import sparse
import subtract_mu_from_sparse  # Import the compiled pybind11 module

def main():
    # 1. Define the sparse matrix X in CSR format
    # Matrix:
    # [[1.0, 0.0, 2.0],
    #  [0.0, 0.0, 3.0],
    #  [4.0, 0.0, 0.0]]
    
    # Define non-zero data, column indices, and row pointers
    data = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float64)  # Non-zero values
    indices = np.array([0, 2, 2, 0], dtype=np.int32)         # Column indices (0-based)
    indptr = np.array([0, 2, 3, 4], dtype=np.int32)          # Row pointers (0-based)
    shape = (3, 3)                                           # Number of rows and columns
    
    # Create a CSR sparse matrix
    X = sparse.csr_matrix((data, indices, indptr), shape=shape)
    
    # Define the Mu vector (bias term for each row)
    Mu = np.array([0.5, 1.5, 2.5], dtype=np.float64)
    
    print("Original X:")
    print(X.toarray())
    
    # 2. Extract data, indices, indptr, and shape from X
    X_data = X.data.astype(np.float64)           # Ensure double precision
    X_indices = X.indices.astype(np.int32)       # Ensure 32-bit integers
    X_indptr = X.indptr.astype(np.int32)         # Ensure 32-bit integers
    X_shape = X.shape                            # Tuple (n1, n2)
    
    # 3. Call the SUBTRACT_MU function
    X_modified_data = subtract_mu_from_sparse.SUBTRACT_MU(
        X_data,
        X_indices,
        X_indptr,
        X_shape,
        Mu
    )
    
    # 4. Reconstruct the modified sparse matrix
    X_modified = sparse.csr_matrix((X_modified_data, X_indices, X_indptr), shape=X_shape)
    
    print("\nModified X after subtracting Mu:")
    print(X_modified.toarray())
    
    # 5. Compute the expected result using SciPy's built-in operations
    # Equivalent to:
    # M = spones(X);
    # X_expected = X - repmat(Mu,1,size(X,2)) .* M;
    # X_expected[X_expected <= 0] = EPS;
    
    EPS = 1e-15
    M = X.copy()
    M.data = np.ones_like(M.data)
    
    # Subtract Mu from non-zero elements based on row indices
    row_indices = np.repeat(np.arange(X_shape[0]), np.diff(X_indptr))
    X_expected_data = X.data - Mu[row_indices]
    
    # Replace non-positive values with EPS
    X_expected_data[X_expected_data <= 0] = EPS
    
    # Construct expected sparse matrix
    X_expected = sparse.csr_matrix((X_expected_data, X.indices, X.indptr), shape=X_shape)
    
    print("\nExpected Modified X:")
    print(X_expected.toarray())
    
    # 6. Compare the outputs
    data_match = np.allclose(X_modified.data, X_expected.data, atol=1e-12)
    indices_match = np.array_equal(X_modified.indices, X_expected.indices)
    indptr_match = np.array_equal(X_modified.indptr, X_expected.indptr)
    
    if data_match and indices_match and indptr_match:
        print("\nTest Passed: X_modified matches expected X.")
    else:
        print("\nTest Failed: X_modified does not match expected X.")
        
        # Detailed discrepancy
        print("\nDifferences in data:")
        print("X_modified.data:", X_modified.data)
        print("X_expected.data:", X_expected.data)
    
if __name__ == "__main__":
    main()


Original X:
[[1. 0. 2.]
 [0. 0. 3.]
 [4. 0. 0.]]

Modified X after subtracting Mu:
[[0.5 0.  1.5]
 [0.  0.  1.5]
 [1.5 0.  0. ]]

Expected Modified X:
[[0.5 0.  1.5]
 [0.  0.  1.5]
 [1.5 0.  0. ]]

Test Passed: X_modified matches expected X.


In [5]:
# Test Case 1: Basic Functionality
# Description: Subtract Mu from a simple sparse matrix X.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU  # Ensure this is the correct import path

class TestSubtractMu(unittest.TestCase):
    def test_basic_functionality(self):
        # Create a simple sparse matrix X in CSR format
        # MATLAB's sparse matrix is in CSC format, but scipy uses CSR by default
        # However, the function should handle this correctly by using row indices
        X_data = np.array([1, 3, 5, 7, 9], dtype=np.double)
        X_indices = np.array([0, 2, 1, 0, 2], dtype=np.int32)
        X_indptr = np.array([0, 2, 3, 5], dtype=np.int32)  # CSR Indptr
        X_shape = (3, 3)
        X = sp.csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)
        
        # Define Mu vector
        Mu = np.array([1, 2, 3], dtype=np.double)
        
        # Expected output using NumPy operations
        # Generate row indices based on indptr
        row_indices = np.repeat(np.arange(X_shape[0]), np.diff(X_indptr))
        # Subtract Mu based on row indices
        X_expected_data = X.data - Mu[row_indices]
        # Replace exact zeros with EPS
        X_expected_data[X_expected_data == 0] = 1e-15
        # Create the expected sparse matrix
        X_expected = sp.csr_matrix((X_expected_data, X.indices, X.indptr), shape=X_shape)
        
        # Call the Python SUBTRACT_MU function
        X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X_shape, Mu)
        X_new = sp.csr_matrix((X_new_data, X.indices, X.indptr), shape=X_shape)
        
        # Convert to dense arrays for comparison
        X_new_dense = X_new.toarray()
        X_expected_dense = X_expected.toarray()
        
        # Display results
        print("Original X:")
        print(X.toarray())
        print("Mu:")
        print(Mu)
        print("Expected X_new:")
        print(X_expected_dense)
        print("Computed X_new:")
        print(X_new_dense)
        
        # Verify correctness
        # Using np.allclose to account for floating-point precision
        self.assertTrue(np.allclose(X_new_dense, X_expected_dense, atol=1e-12),
                        "Test Case 1 Failed: Output does not match expected result.")
        print("Test Case 1 Passed: Basic Functionality")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.005s

OK


Original X:
[[1. 0. 3.]
 [0. 5. 0.]
 [7. 0. 9.]]
Mu:
[1. 2. 3.]
Expected X_new:
[[1.e-15 0.e+00 2.e+00]
 [0.e+00 3.e+00 0.e+00]
 [4.e+00 0.e+00 6.e+00]]
Computed X_new:
[[1.e-15 0.e+00 2.e+00]
 [0.e+00 3.e+00 0.e+00]
 [4.e+00 0.e+00 6.e+00]]
Test Case 1 Passed: Basic Functionality


In [4]:
# Test Case 2: Zero Matrix
# Description: Subtract Mu from a zero sparse matrix X.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU

class TestSubtractMu(unittest.TestCase):
    def test_zero_matrix(self):
        # Create a zero sparse matrix X in CSR format
        X_data = np.array([], dtype=np.double)
        X_indices = np.array([], dtype=np.int32)
        X_indptr = np.array([0, 0, 0, 0], dtype=np.int32)
        X_shape = (3, 3)
        X = sp.csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)
        
        # Define Mu vector
        Mu = np.array([1, 2, 3], dtype=np.double)
        
        # Expected output is a zero sparse matrix
        X_expected = X.copy()
        
        # Call the Python function
        X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X_shape, Mu)
        X_new = sp.csr_matrix((X_new_data, X.indices, X_indptr), shape=X_shape)
        
        # Convert to dense arrays for comparison
        X_new_dense = X_new.toarray()
        X_expected_dense = X_expected.toarray()
        
        # Display results
        print("Original X (Zero Matrix):")
        print(X.toarray())
        print("Mu:")
        print(Mu)
        print("Expected X_new:")
        print(X_expected_dense)
        print("Computed X_new:")
        print(X_new_dense)
        
        # Verify correctness
        self.assertTrue(np.array_equal(X_new_dense, X_expected_dense), "Test Case 2 Failed: Output does not match expected result.")
        print("Test Case 2 Passed: Zero Matrix")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


Original X (Zero Matrix):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Mu:
[1. 2. 3.]
Expected X_new:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Computed X_new:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Test Case 2 Passed: Zero Matrix


In [3]:
# Test Case 3: Mu Contains Zero Entries
# Description: Some elements in Mu are zero.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU

class TestSubtractMu(unittest.TestCase):
    def test_mu_with_zero_entries(self):
        # Create a simple sparse matrix X in CSR format
        X_data = np.array([1, 3, 5, 7, 9], dtype=np.double)
        X_indices = np.array([0, 2, 1, 0, 2], dtype=np.int32)
        X_indptr = np.array([0, 2, 3, 5], dtype=np.int32)
        X_shape = (3, 3)
        X = sp.csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)
        
        # Define Mu vector with zero entries
        Mu = np.array([0, 2, 0], dtype=np.double)
        
        # Expected output using NumPy operations
        X_expected_data = X.data - Mu[X.indices]
        # Replace exact zeros with EPS
        X_expected_data[X_expected_data == 0] = 1e-15
        X_expected = sp.csr_matrix((X_expected_data, X.indices, X.indptr), shape=X_shape)
        
        # Call the Python function
        X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X_shape, Mu)
        X_new = sp.csr_matrix((X_new_data, X.indices, X.indptr), shape=X_shape)
        
        # Convert to dense arrays for comparison
        X_new_dense = X_new.toarray()
        X_expected_dense = X_expected.toarray()
        
        # Display results
        print("Original X:")
        print(X.toarray())
        print("Mu with zeros:")
        print(Mu)
        print("Expected X_new:")
        print(X_expected_dense)
        print("Computed X_new:")
        print(X_new_dense)
        
        # Verify correctness
        self.assertTrue(np.array_equal(X_new_dense, X_expected_dense), "Test Case 3 Failed: Output does not match expected result.")
        print("Test Case 3 Passed: Mu Contains Zero Entries")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


Original X:
[[1. 0. 3.]
 [0. 5. 0.]
 [7. 0. 9.]]
Mu with zeros:
[0. 2. 0.]
Expected X_new:
[[1. 0. 3.]
 [0. 3. 0.]
 [7. 0. 9.]]
Computed X_new:
[[1. 0. 3.]
 [0. 3. 0.]
 [7. 0. 9.]]
Test Case 3 Passed: Mu Contains Zero Entries


In [2]:
# Test Case 4: Mismatched Dimensions (Error Handling)
# Description: Mu has a different number of rows than X.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU  # Ensure this matches your compiled module's name

class TestSubtractMu(unittest.TestCase):
    def test_mismatched_dimensions(self):
        # Create a simple sparse matrix X in CSR format with 3 rows and 2 columns
        X_data = np.array([1, 5, 7], dtype=np.double)
        X_indices = np.array([0, 1, 0], dtype=np.int32)
        X_indptr = np.array([0, 1, 2, 3], dtype=np.int32)  # CSR Indptr for 3 rows
        X_shape = (3, 2)
        X = sp.csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)
        
        # Define Mu vector with mismatched dimensions (only 2 elements instead of 3)
        Mu = np.array([1, 2], dtype=np.double)
        
        # Attempt to call the SUBTRACT_MU function and expect a ValueError
        with self.assertRaises(ValueError) as context:
            # This should raise a ValueError due to Mu length mismatch
            X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X_shape, Mu)
            X_new = sp.csr_matrix((X_new_data, X.indices, X.indptr), shape=X_shape)
        
        # Check if the exception message is as expected
        self.assertIn("Mu vector length does not match the number of rows in X", str(context.exception),
                      "Test Case 4 Failed: Incorrect error message.")
        
        # Print confirmation that the correct exception was raised
        print("Test Case 4 Passed: Mismatched Dimensions Error Handling")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


Test Case 4 Passed: Mismatched Dimensions Error Handling


In [10]:
# Test Case 5: Mu with Negative Entries
# Description: Mu contains negative values.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU

class TestSubtractMu(unittest.TestCase):
    def test_mu_with_negative_entries(self):
        # Create a simple sparse matrix X in CSR format
        X_data = np.array([2, 4, 6, 8, 10], dtype=np.double)
        X_indices = np.array([0, 2, 1, 0, 2], dtype=np.int32)
        X_indptr = np.array([0, 2, 3, 5], dtype=np.int32)
        X_shape = (3, 3)
        X = sp.csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)
        
        # Define Mu vector with negative entries
        Mu = np.array([-1, 2, -3], dtype=np.double)
        
        # Generate row indices based on indptr
        row_indices = np.repeat(np.arange(X_shape[0]), np.diff(X_indptr))
        
        # Expected output using NumPy operations
        X_expected_data = X.data - Mu[row_indices]
        # Replace exact zeros with EPS
        X_expected_data[X_expected_data == 0] = 1e-15
        X_expected = sp.csr_matrix((X_expected_data, X.indices, X.indptr), shape=X_shape)
        
        # Call the Python SUBTRACT_MU function
        X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X_shape, Mu)
        X_new = sp.csr_matrix((X_new_data, X.indices, X.indptr), shape=X_shape)
        
        # Convert to dense arrays for comparison
        X_new_dense = X_new.toarray()
        X_expected_dense = X_expected.toarray()
        
        # Display results
        print("Original X:")
        print(X.toarray())
        print("Mu with Negative Entries:")
        print(Mu)
        print("Expected X_new:")
        print(X_expected_dense)
        print("Computed X_new:")
        print(X_new_dense)
        
        # Verify correctness
        # Using np.allclose to account for floating-point precision
        self.assertTrue(np.allclose(X_new_dense, X_expected_dense, atol=1e-12),
                        "Test Case 5 Failed: Output does not match expected result.")
        print("Test Case 5 Passed: Mu with Negative Entries")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.006s

OK


Original X:
[[ 2.  0.  4.]
 [ 0.  6.  0.]
 [ 8.  0. 10.]]
Mu with Negative Entries:
[-1.  2. -3.]
Expected X_new:
[[ 3.  0.  5.]
 [ 0.  4.  0.]
 [11.  0. 13.]]
Computed X_new:
[[ 3.  0.  5.]
 [ 0.  4.  0.]
 [11.  0. 13.]]
Test Case 5 Passed: Mu with Negative Entries


In [11]:
# Test Case 6: Large Sparse Matrix
# Description: X is a large sparse matrix.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU  # Ensure this matches your compiled module's name

class TestSubtractMu(unittest.TestCase):
    def test_large_sparse_matrix(self):
        # Define matrix size
        n1 = 1000
        n2 = 1000
        density = 0.01  # 1% non-zero entries

        # Generate random sparse matrix X with density 0.01
        np.random.seed(0)  # For reproducibility
        X = sp.random(n1, n2, density=density, format='csr', data_rvs=np.random.randn)

        # Define Mu vector
        Mu = np.random.randn(n1)

        # Generate row indices based on indptr
        row_indices = np.repeat(np.arange(X.shape[0]), np.diff(X.indptr))

        # Expected output using NumPy operations
        X_expected_data = X.data - Mu[row_indices]
        # Replace exact zeros with EPS
        X_expected_data[X_expected_data == 0] = 1e-15
        X_expected = sp.csr_matrix((X_expected_data, X.indices, X.indptr), shape=X.shape)

        # Call the Python SUBTRACT_MU function
        X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X.shape, Mu)
        X_new = sp.csr_matrix((X_new_data, X.indices, X.indptr), shape=X.shape)

        # Convert to dense arrays for comparison
        X_new_dense = X_new.toarray()
        X_expected_dense = X_expected.toarray()

        # Display sample results for verification
        # (Optional: Comment out in actual unit tests to reduce verbosity)
        print("Original X Sample (First 5 Rows):")
        print(X.toarray()[:5, :5])  # Display a small sample
        print("\nMu Sample (First 5 Elements):")
        print(Mu[:5])
        print("\nExpected X_new Sample (First 5 Rows):")
        print(X_expected_dense[:5, :5])
        print("\nComputed X_new Sample (First 5 Rows):")
        print(X_new_dense[:5, :5])

        # Verify correctness using allclose with a tight tolerance
        self.assertTrue(np.allclose(X_new_dense, X_expected_dense, atol=1e-12),
                        "Test Case 6 Failed: Output does not match expected result.")
        print("Test Case 6 Passed: Large Sparse Matrix")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.055s

OK


Original X Sample (First 5 Rows):
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

Mu Sample (First 5 Elements):
[ 0.52228701 -0.48033045 -0.40823566 -0.36652602 -0.35335626]

Expected X_new Sample (First 5 Rows):
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

Computed X_new Sample (First 5 Rows):
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
Test Case 6 Passed: Large Sparse Matrix


In [13]:
# Test Case 7: All Mu Values Result in Zero After Subtraction
# Description: Subtracting Mu from X results in zeros, which should be replaced by EPS.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU  # Ensure this matches your compiled module's name

class TestSubtractMu(unittest.TestCase):
    def test_all_mu_values_result_in_zero_after_subtraction(self):
        # Create a simple sparse matrix X in CSR format
        X_data = np.array([1, 2, 3, 4], dtype=np.double)
        X_indices = np.array([0, 1, 0, 1], dtype=np.int32)
        X_indptr = np.array([0, 2, 4], dtype=np.int32)  # CSR Indptr for 2 rows
        X_shape = (2, 2)
        X = sp.csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)
        
        # Define Mu vector such that X - Mu = 0
        Mu = np.array([1, 2], dtype=np.double)  # Length should match number of rows in X
        
        # Generate row indices based on indptr
        # For CSR format, row_indices can be generated by repeating row numbers based on number of non-zeros per row
        row_indices = np.repeat(np.arange(X_shape[0]), np.diff(X.indptr))
        
        # Expected output using NumPy operations
        X_expected_data = X.data - Mu[row_indices]
        # Replace exact zeros with EPS
        EPS = 1e-15
        X_expected_data[X_expected_data == 0] = EPS
        # Create the expected sparse matrix
        X_expected = sp.csr_matrix((X_expected_data, X.indices, X.indptr), shape=X_shape)
        
        # Call the Python SUBTRACT_MU function
        X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X_shape, Mu)
        # Reconstruct the new sparse matrix from the returned data
        X_new = sp.csr_matrix((X_new_data, X.indices, X.indptr), shape=X_shape)
        
        # Convert sparse matrices to dense arrays for comparison
        X_new_dense = X_new.toarray()
        X_expected_dense = X_expected.toarray()
        
        # Display results
        print("Original X:")
        print(X.toarray())
        print("\nMu:")
        print(Mu)
        print("\nExpected X_new (with EPS):")
        print(X_expected_dense)
        print("\nComputed X_new:")
        print(X_new_dense)
        
        # Verify correctness using np.allclose to account for floating-point precision
        self.assertTrue(np.allclose(X_new_dense, X_expected_dense, atol=1e-12),
                        "Test Case 7 Failed: Output does not match expected result.")
        print("Test Case 7 Passed: All Mu Values Result in Zero After Subtraction")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.012s

OK


Original X:
[[1. 2.]
 [3. 4.]]

Mu:
[1. 2.]

Expected X_new (with EPS):
[[1.e-15 1.e+00]
 [1.e+00 2.e+00]]

Computed X_new:
[[1.e-15 1.e+00]
 [1.e+00 2.e+00]]
Test Case 7 Passed: All Mu Values Result in Zero After Subtraction


In [14]:
# Test Case 8: Empty Sparse Matrix
# Description: X is an empty sparse matrix.

import unittest
import numpy as np
import scipy.sparse as sp
from subtract_mu_from_sparse import SUBTRACT_MU

class TestSubtractMu(unittest.TestCase):
    def test_empty_sparse_matrix(self):
        # Create an empty sparse matrix X in CSR format
        X_data = np.array([], dtype=np.double)
        X_indices = np.array([], dtype=np.int32)
        X_indptr = np.array([0, 0, 0, 0], dtype=np.int32)
        X_shape = (3, 3)
        X = sp.csr_matrix((X_data, X_indices, X_indptr), shape=X_shape)
        
        # Define Mu vector
        Mu = np.array([1, 2, 3], dtype=np.double)
        
        # Expected output is an empty sparse matrix
        X_expected = X.copy()
        
        # Call the Python function
        X_new_data = SUBTRACT_MU(X.data, X.indices, X.indptr, X_shape, Mu)
        X_new = sp.csr_matrix((X_new_data, X.indices, X.indptr), shape=X_shape)
        
        # Convert to dense arrays for comparison
        X_new_dense = X_new.toarray()
        X_expected_dense = X_expected.toarray()
        
        # Display results
        print("Original X (Empty Matrix):")
        print(X.toarray())
        print("Mu:")
        print(Mu)
        print("Expected X_new:")
        print(X_expected_dense)
        print("Computed X_new:")
        print(X_new_dense)
        
        # Verify correctness
        self.assertTrue(np.array_equal(X_new_dense, X_expected_dense), "Test Case 8 Failed: Output does not match expected result.")
        print("Test Case 8 Passed: Empty Sparse Matrix")

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


Original X (Empty Matrix):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Mu:
[1. 2. 3.]
Expected X_new:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Computed X_new:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Test Case 8 Passed: Empty Sparse Matrix
